# Charlotteburg Reservoir contributing watershed and USGS stations

Adapted from Module6_CanistearWatershed_USGS.ipynb, following the same procedure: retrieve the USGS NLDI unsimplified upstream basin for reference station **01382380**, project to UTM 18N, create a **2 km outward buffer**, discover all USGS station types and statuses, apply exact spatial filtering, and export GIS layers and an interactive map.

The map uses **Esri World Street Map**, green basin stations and orange buffer-only stations, with separate watershed/buffer controls and USGS popups. It displays directly in Colab.

Published reference drainage area: **56.2 mi²**. The notebook computes the area discrepancy and station counts on execution; it does not assume the Canistear results apply here. This is NLDI basin retrieval, not a new DEM delineation. Nearby sites are not assumed to drain to the reservoir. Operational transfers are excluded.


## 1. Environment
Run cells in order. Install the public Python dependencies below. Outputs are written to data/charlotteburg under the repository root when available, or the current working directory in a standalone Colab session. QGIS regeneration is optional.


## Reuse this workflow
The study settings below select Charlotteburg. Follow the same retrieval, metric buffer, spatial filtering, export, and map steps as Module 6. Station counts and areas are calculated from the retrieved data.


In [1]:
%pip install -q requests pandas geopandas shapely pyproj pyogrio
from pathlib import Path


## 2. Retrieve basin, build buffer, discover all stations, and export
EPSG:32618 (UTM zone 18N) is used for the 2,000 m buffer, distance, and area operations. GeoJSON and interactive map coordinates are WGS84. GeoPackage layers retain UTM coordinates. The public NWIS query covers the buffer bounding box, then exact geometric filtering removes outside points. Failed requests raise errors rather than silently generating a zero-station result.

The HTML map uses Leaflet with the same Esri World Street Map provider as Module 5. Internet is required for JavaScript and basemap tiles; the GeoPackage and QGIS vector layers work offline.

### Imports

In [2]:
from pathlib import Path
import json, io, datetime, html
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape



### Study configuration

In [3]:
# Reusable study settings: change these for another USGS outlet.
OUTLET_SITE = '01382380'
STUDY_NAME = 'Charlotteburg Reservoir contributing watershed'
BUFFER_DISTANCE_KM = 2.0
PROJECTED_CRS = 32618  # UTM 18N; select the local metric CRS for another region.
PUBLISHED_AREA_SQMI = 56.2  # Optional independent area check; update for another site.



### Output folder and public-service request helper

In [4]:
# Locate the project root in a clone; Colab defaults to its current working directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'charlotteburg').is_dir()), Path.cwd())
OUT = ROOT / 'data' / 'charlotteburg'
OUT.mkdir(parents=True, exist_ok=True)
RAW = OUT / 'sources'; RAW.mkdir(exist_ok=True)
session = requests.Session()
log = []
def fetch(url, params, name):
    r = session.get(url, params=params, timeout=90); r.raise_for_status()
    (RAW / name).write_text(r.text, encoding='utf-8')
    log.append({'url':r.url,'retrieved_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'file':name})
    return r



### Retrieve the contributing watershed and check its area

In [5]:
basin_json = fetch(f'https://api.water.usgs.gov/nldi/linked-data/nwissite/USGS-{OUTLET_SITE}/basin', {'simplified':'false'}, 'nldi_basin.geojson').json()
basin = gpd.GeoDataFrame.from_features(basin_json['features'], crs=4326).to_crs(PROJECTED_CRS)
basin = gpd.GeoDataFrame({'name':[STUDY_NAME]}, geometry=[basin.geometry.union_all()], crs=PROJECTED_CRS)
area = basin.area.iloc[0]/1e6
basin['area_km2'] = area; basin['area_sqmi'] = area/2.589988110336

assert not basin.empty and basin.geometry.is_valid.all() and area > 0, 'Invalid or empty NLDI basin; review outlet before continuing.'
print(f'Outlet: USGS-{OUTLET_SITE}; basin area: {area:.2f} km² ({area/2.589988110336:.2f} mi²)')


Outlet: USGS-01382380; basin area: 147.45 km² (56.93 mi²)


### Create the metric search buffer

In [6]:
buffer = basin.copy(); buffer['name'] = f'{BUFFER_DISTANCE_KM:g} km watershed search buffer'; buffer.geometry = basin.buffer(BUFFER_DISTANCE_KM * 1000)
ring = buffer.copy(); ring['name'] = f'{BUFFER_DISTANCE_KM:g} km buffer outside watershed'; ring.geometry = buffer.geometry.difference(basin.geometry.iloc[0])


### Retrieve the USGS station inventory

In [7]:
# Use the USGS monitoring-location API; the legacy NWIS host can stall in Colab.
from concurrent.futures import ThreadPoolExecutor
w,s,e,n = buffer.to_crs(4326).total_bounds
url = 'https://api.waterdata.usgs.gov/ogcapi/v0/collections/monitoring-locations/items'
params = {'f':'json','bbox':f'{w},{s},{e},{n}','limit':10000}
features = []
request_pool = ThreadPoolExecutor(max_workers=1)
try:
    while url:
        print('Retrieving USGS monitoring locations...', flush=True)
        response = request_pool.submit(requests.get, url, params=params, timeout=(10,30)).result(timeout=45)
        response.raise_for_status()
        payload = response.json()
        name = f'usgs_monitoring_locations_{len(log)}.geojson'
        (RAW / name).write_text(response.text, encoding='utf-8')
        log.append({'url':response.url,'retrieved_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'file':name})
        features.extend(payload['features'])
        url = next((link['href'] for link in payload.get('links',[]) if link.get('rel') == 'next'), None)
        params = None
finally:
    request_pool.shutdown(wait=False, cancel_futures=True)
assert features, 'USGS returned no monitoring locations.'
records = []
for feature in features:
    p = feature['properties']
    coords = feature['geometry']['coordinates']
    records.append({'agency_cd':p['agency_code'], 'site_no':p['monitoring_location_number'],
                    'station_nm':p['monitoring_location_name'], 'site_tp_cd':p['site_type_code'],
                    'dec_long_va':coords[0], 'dec_lat_va':coords[1]})
df = pd.DataFrame(records).drop_duplicates('site_no')
assert df[['dec_lat_va','dec_long_va']].notna().all().all(), 'Missing coordinates require review.'
print(f'Retrieved {len(df)} USGS sites in bounding box; no site-type or status filter applied.')


Retrieving USGS monitoring locations...
Retrieved 70 USGS sites in bounding box; no site-type or status filter applied.


### Filter, classify, and calculate distances

In [8]:
sites = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.dec_long_va,df.dec_lat_va), crs=4326).to_crs(PROJECTED_CRS)
sites = sites[sites.geometry.intersects(buffer.geometry.iloc[0])].copy().drop_duplicates('site_no')
sites['location_type'] = ['watershed' if basin.geometry.iloc[0].covers(p) else 'buffer_only' for p in sites.geometry]
sites['distance_to_basin_km'] = sites.distance(basin.geometry.iloc[0])/1000
sites['distance_to_centroid_km'] = sites.distance(basin.geometry.iloc[0].centroid)/1000
sites['usgs_url'] = 'https://waterdata.usgs.gov/monitoring-location/USGS-'+sites.site_no+'/'
sites = sites.sort_values(['location_type','site_no'])


### Export GIS layers, catalog, and provenance

In [9]:
layers = {'watershed':basin,'search_buffer':buffer,'buffer_only_zone':ring,'usgs_sites':sites,
          'sites_in_basin':sites[sites.location_type=='watershed'].copy(), 'sites_buffer_only':sites[sites.location_type=='buffer_only'].copy()}
gpkg = OUT/'Charlotteburg.gpkg'
if gpkg.exists(): gpkg.unlink()
for name, frame in layers.items():
    frame.to_file(gpkg,layer=name,driver='GPKG')
    frame.to_crs(4326).to_file(OUT/(name+'.geojson'),driver='GeoJSON')
sites.drop(columns='geometry').to_csv(OUT/'usgs_station_catalog.csv',index=False)
summary = {'watershed_area_km2':area,'watershed_area_sqmi':area/2.589988110336,'published_usgs_area_sqmi':PUBLISHED_AREA_SQMI,
 'basin_sites':int((sites.location_type=='watershed').sum()),'buffer_only_sites':int((sites.location_type=='buffer_only').sum()),
 'total_sites':len(sites),'buffer_km':BUFFER_DISTANCE_KM,'crs':f'EPSG:{PROJECTED_CRS}','method':f'USGS NLDI unsimplified upstream basin for USGS-{OUTLET_SITE}; not a new DEM delineation',
 'scope':'All NWIS site types, all statuses, no parameter filter; exact polygon intersection after bounding-box query',
 'area_difference_pct':100*((area/2.589988110336)/PUBLISHED_AREA_SQMI-1),
 'sources':log}
(OUT/'metadata.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps({k:v for k,v in summary.items() if k!='sources'},indent=2))



{
  "watershed_area_km2": 147.44513814843344,
  "watershed_area_sqmi": 56.928886105699284,
  "published_usgs_area_sqmi": 56.2,
  "basin_sites": 23,
  "buffer_only_sites": 23,
  "total_sites": 46,
  "buffer_km": 2.0,
  "crs": "EPSG:32618",
  "method": "USGS NLDI unsimplified upstream basin for USGS-01382380; not a new DEM delineation",
  "scope": "All NWIS site types, all statuses, no parameter filter; exact polygon intersection after bounding-box query",
  "area_difference_pct": 1.296950366012961
}


### Create the interactive map

In [10]:
# Module 5-style interactive map, with separate station groups and USGS popups.
data = {k:json.loads(v.to_crs(4326).to_json()) for k,v in layers.items() if k in ['watershed','search_buffer','sites_in_basin','sites_buffer_only']}
page = '''<!doctype html><html><head><meta charset="utf-8"><title>Charlotteburg Reservoir watershed and USGS stations</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"><script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>body{margin:0;font:15px Arial}#map{height:88vh}header{padding:12px 20px}h2{margin:0 0 6px} .legend{background:white;padding:12px;line-height:1.6}</style></head><body>
<header><h2>Charlotteburg Reservoir contributing watershed</h2>AREA km² · INSIDE sites in basin · OUTSIDE sites in BUFFER_LABEL buffer only · All USGS site types and statuses</header><div id="map"></div>
<script>const data=DATA; const map=L.map('map');
L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',{attribution:'Tiles © Esri and the GIS User Community'}).addTo(map);
const buf=L.geoJSON(data.search_buffer,{style:{color:'#3388cc',weight:1,fillOpacity:.16}}).addTo(map);
const basin=L.geoJSON(data.watershed,{style:{color:'#173d89',weight:3,fillOpacity:.35}}).addTo(map);
function safe(s){const d=document.createElement('div');d.textContent=s||'';return d.innerHTML}
function points(fc,color){return L.geoJSON(fc,{pointToLayer:(f,ll)=>L.circleMarker(ll,{radius:5,color:'#fff',weight:1,fillColor:color,fillOpacity:.9}),onEachFeature:(f,l)=>{const p=f.properties;l.bindTooltip(p.site_no+' '+p.station_nm);l.bindPopup('<b>USGS '+safe(p.site_no)+'</b><br>'+safe(p.station_nm)+'<br>Type: '+safe(p.site_tp_cd)+'<br>Zone: '+safe(p.location_type)+'<br>Distance to basin: '+Number(p.distance_to_basin_km).toFixed(2)+' km<br><a target="_blank" rel="noopener" href="'+p.usgs_url+'">USGS station page</a>')}}).addTo(map)}
const inside=points(data.sites_in_basin,'#16834b'),outside=points(data.sites_buffer_only,'#ed8b23');
L.control.layers(null,{'BUFFER_LABEL search buffer':buf,'Contributing watershed':basin,'USGS sites in basin':inside,'USGS sites in buffer only':outside}).addTo(map);
map.fitBounds(buf.getBounds());const legend=L.control({position:'bottomleft'});legend.onAdd=()=>{const d=L.DomUtil.create('div','legend');d.innerHTML='● USGS site in basin<br>● USGS site in buffer only<br>Blue: watershed and BUFFER_LABEL buffer<br>Source: USGS NLDI / NWIS';return d};legend.addTo(map);</script></body></html>'''
page = page.replace('Charlotteburg Reservoir contributing watershed', html.escape(STUDY_NAME)).replace('BUFFER_LABEL', f'{BUFFER_DISTANCE_KM:g} km').replace('AREA',f'{area:.2f}').replace('INSIDE',str(summary['basin_sites'])).replace('OUTSIDE',str(summary['buffer_only_sites'])).replace('DATA',json.dumps(data))
(OUT/'Charlotteburg_map.html').write_text(page,encoding='utf-8')



59872

## 3. View the station catalog and interactive map
Run after the preceding retrieval and export cells. IDs are read as text to preserve leading zeros.


In [11]:
from IPython.display import display, HTML
catalog = pd.read_csv(OUT / 'usgs_station_catalog.csv', dtype={'site_no': str})
display(catalog[['site_no', 'station_nm', 'site_tp_cd', 'location_type']])
page = (OUT / 'Charlotteburg_map.html').read_text(encoding='utf-8')
display(HTML('<iframe title="Charlotteburg watershed and USGS stations" style="width:100%;height:750px;border:0" srcdoc="' + html.escape(page, quote=True) + '"></iframe>'))


,site_no,station_nm,site_tp_cd,location_type
0,01368823,Lake Lookout Brook Trib near Kempe NJ,ST,buffer_only
1,01368935,Town Brook tributary near Highland Lakes NJ,ST,buffer_only
2,01379615,Russia Brook at Russia NJ,ST,buffer_only
3,01382385,Pequannock River at Green Pond Junction NJ,ST,buffer_only
4,01382392,Echo Lake 4800' upstream of dam near Macopin NJ,LK,buffer_only
5,01382396,Echo Lake 1500' upstream of dam near Macopin NJ,LK,buffer_only
6,01382400,Echo Lk near Charlotteburg NJ,LK,buffer_only
7,01382410,Macopin River at Echo Lake NJ,ST,buffer_only
8,01382430,Mathews Brook diversion near Macopin NJ,ST,buffer_only
9,01382450,Macopin R at Macopin Re NJ,ST,buffer_only


/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## 4. Optional native QGIS project regeneration
Run the function below only in a QGIS Python environment to create the project and vector PNG. Skip this step in Colab.


In [12]:
def make_qgis():
    import os
    dll_handles = []
    if os.name == 'nt':
        for directory in ['C:/Program Files/QGIS 3.44.1/bin','C:/Program Files/QGIS 3.44.1/apps/Qt5/bin','C:/Program Files/QGIS 3.44.1/apps/qgis/bin']:
            if Path(directory).exists(): dll_handles.append(os.add_dll_directory(directory))
    from qgis.core import (QgsApplication,QgsProject,QgsVectorLayer,QgsCoordinateReferenceSystem,QgsFillSymbol,QgsMarkerSymbol,QgsRasterLayer,QgsRectangle,QgsMapSettings,QgsMapRendererParallelJob)
    from qgis.PyQt.QtCore import QSize
    from qgis.PyQt.QtGui import QColor
    app=QgsApplication.instance()
    if app is None:
        app=QgsApplication([],False);app.initQgis()
    project=QgsProject.instance();project.clear();project.setCrs(QgsCoordinateReferenceSystem(f'EPSG:{PROJECTED_CRS}'))
    project.setFileName(str(OUT/'Charlotteburg.qgs'))
    project.setTitle('Charlotteburg Reservoir contributing watershed and all USGS sites')
    basemap=QgsRasterLayer('type=xyz&url=https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/%7Bz%7D/%7By%7D/%7Bx%7D','Esri World Street Map (internet required)','wms')
    project.addMapLayer(basemap)
    qlayers=[]
    styles=[('search_buffer',f'{BUFFER_DISTANCE_KM:g} km search buffer','110,180,235,45','50,110,190,255'),('watershed',STUDY_NAME,'30,65,145,95','20,50,125,255'),('sites_buffer_only',f'USGS buffer only ({summary["buffer_only_sites"]})','#ed8b23',None),('sites_in_basin',f'USGS in basin ({summary["basin_sites"]})','#16834b',None)]
    for key,title,fill,stroke in styles:
        layer=QgsVectorLayer(str(gpkg)+'|layername='+key,title,'ogr');assert layer.isValid(),key
        symbol=QgsFillSymbol.createSimple({'color':fill,'outline_color':stroke,'outline_width':'0.6'}) if stroke else QgsMarkerSymbol.createSimple({'name':'circle','color':fill,'outline_color':'white','size':'2.5'})
        layer.renderer().setSymbol(symbol);project.addMapLayer(layer);qlayers.insert(0,layer)
    extent=QgsRectangle(*buffer.total_bounds);extent.scale(1.06)
    project.viewSettings().setDefaultViewExtent(__import__('qgis.core',fromlist=['QgsReferencedRectangle']).QgsReferencedRectangle(extent,project.crs()))
    assert project.write(), 'Project save failed'
    # Export an offline map: vector data remain readable without external tiles.
    settings=QgsMapSettings();settings.setLayers(qlayers);settings.setDestinationCrs(project.crs());settings.setExtent(extent);settings.setOutputSize(QSize(1600,1400));settings.setBackgroundColor(QColor('#f4f7fa'))
    job=QgsMapRendererParallelJob(settings);job.start();job.waitForFinished();job.renderedImage().save(str(OUT/'Charlotteburg_map.png'))
    # Reopen to verify relative paths and layer validity.
    project.clear();assert project.read(str(OUT/'Charlotteburg.qgs'))
    assert all(x.isValid() for x in project.mapLayers().values()), 'Invalid project layer'
    print('QGIS project saved and reopened; all layers valid.')


# make_qgis()  # Uncomment only in a QGIS Python environment.


## 5. Outputs and interpretation
Outputs include Charlotteburg.gpkg, GeoJSON layers, usgs_station_catalog.csv, metadata.json and Charlotteburg_map.html in data/charlotteburg. The optional QGIS function creates Charlotteburg.qgs and an offline vector PNG when run inside QGIS.

Green points are inside the contributing watershed; orange points are within the 2 km outside buffer. The map uses Esri tiles and requires internet. Station spatial inclusion does not establish current measurement availability, telemetry frequency, rainfall suitability, or hydraulic connectivity.

Review the calculated area discrepancy against 56.2 mi² before engineering use. Existing station_analysis.csv remains a separate operational inventory snapshot and is not refreshed by this spatial workflow.
